# UCI 客户价值 Mart 审计

步骤 10 的可复现、只读审计伴侣。

## tl;dr

客户价值 Mart 覆盖 4,338 名可识别客户、18,532 个订单（92.845691% 有效订单覆盖）和 £8,887,208.89 历史消费。历史价值是累计消费代理，不是未来 LTV 预测。

## Context & Methods

### Key Assumptions

- 只使用 `uci_order_v1` 的有客户 ID 有效订单。
- 快照日为最后订单日后一天：2011-12-10。
- RFM 分数使用并列值一致的 `CUME_DIST` 五分位。
- 分群仅用于公开样例的可解释演示，不代表真实客户状态。

## Data

来源批次：`7aa2b0f4-12aa-4dfb-890c-ffadc8f9d6fa`；最新修正构建运行：`5ad2c2ef-2368-4436-8041-607c908ab10e`；定义版本：`uci_customer_value_v1`。

In [1]:
from pathlib import Path
import os

import pandas as pd
import pymysql
from dotenv import load_dotenv

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(project_root / '.env')
batch_id = '7aa2b0f4-12aa-4dfb-890c-ffadc8f9d6fa'
definition_version = 'uci_customer_value_v1'

connection = pymysql.connect(host=os.environ.get('MYSQL_HOST', '127.0.0.1'), port=int(os.environ.get('MYSQL_PORT', '3306')), user=os.environ.get('MYSQL_USER', 'root'), password=os.environ['MYSQL_ROOT_PASSWORD'], charset='utf8mb4')

## Results

In [2]:
coverage_summary = pd.read_sql(
    '''SELECT snapshot_date, valid_orders, identified_orders, identified_order_coverage,
              identified_customers, repeat_customers, historical_value_gbp
       FROM insightops_mart.uci_customer_value_summary
       WHERE source_batch_id = %s AND definition_version = %s''',
    connection, params=(batch_id, definition_version)
)
coverage_summary

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_5829/3384641736.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  coverage_summary = pd.read_sql(


,snapshot_date,valid_orders,identified_orders,identified_order_coverage,identified_customers,repeat_customers,historical_value_gbp
0,2011-12-10,19960,18532,0.928457,4338,2845,8887208.894


In [3]:
segment_summary = pd.read_sql(
    '''SELECT rfm_segment, COUNT(*) AS customers,
              ROUND(SUM(historical_value_gbp), 2) AS historical_value_gbp
       FROM insightops_mart.uci_customer_value_fact
       WHERE source_batch_id = %s AND definition_version = %s
       GROUP BY rfm_segment ORDER BY customers DESC''',
    connection, params=(batch_id, definition_version)
)
segment_summary

/var/folders/vx/ctbyqz_x2g7169l40g5971w40000gn/T/ipykernel_5829/2018192409.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  segment_summary = pd.read_sql(


,rfm_segment,customers,historical_value_gbp
0,STANDARD,1719,1013035.46
1,CHAMPION,1006,5848963.35
2,AT_RISK,765,919286.91
3,LOYAL,612,1029722.52
4,NEW_OR_PROMISING,236,76200.65


## Takeaways

后续异常检测或报告可使用 RFM 分群进行描述性切片，但任何未来价值或流失结论需要单独的时间切分、模型验证和评估步骤。